# RCA Cause-Tag Behaviour + Control-Action Details — one sheet per alarm tag

For every configured alarm tag, take its **clustered** RCA file `DATA/RCA_<full>_clustered_top5.csv`
(one row per **raw** alarm cluster, with its top-5 RCA cause tags), keep **only the use-case episodes**,
and enrich each, per episode:

1. **Cause-tag pre-alarm behaviour** — for each `Cause 1–5` tag, three SME-aligned descriptors of its
   PV over the **3 h before the alarm**, written into the cause cell as `pattern | direction | ROC`
   (e.g. `03PIC_1013.PV [ramp | increasing | slow]`):
   - **pattern** — `flat` / `oscillatory` / `step` / `sudden_drift` / `ramp` (SME "Pattern per Cause")
   - **direction** — `increasing` / `decreasing` / `no clear direction` (SME "Direction per Cause")
   - **ROC** — `stable` / `slow` / `fast`, the speed of movement as % of the operating band per hour
     (SME "Behavior per Cause – Slow/Fast"). Oscillation is detected from real peaks/troughs +
     total-variation (not net slope), so large in-band swings are no longer mislabelled as stagnant.
2. **Control-action details** — operator OP/SP/MODE moves in the alarm window, KG-filtered to tags
   knowledge-graph-relevant to that alarm tag:
   - `KG_operated_tags` / `eliminated_tags`
   - `action_details` — per KG-kept tag, chronological, direction ↑/↓ + net step (OP %, SP EU)
   - per tag-type **`… dir`** + **`… avg|step|`** columns (direction + average step magnitude)

## Episode selection (which clusters we keep)
The clustered RCA CSV holds **every** raw cluster. We subset it to the clusters flagged
**`filtered=True`** in the `control_actions` sheet of each tag's **31-Jul pipeline workbook**. That flag
marks the clusters that survived the pipeline funnel used for these runs:
**Stage 1** (duration ≥ 2 min) → **Stage 2** (FI1000 `02FI_1000.PV` in **[7, 9]**, 0 % expansion) →
**Stage 4** (has OP/SP/MODE control actions). **Stage 3** (pre-alarm quiet window) was **disabled** for
these 31-Jul runs. Join: `floor(cluster_start_time, 'min')` == clustered-CSV `Alarm_Timestamp`.

**The clustered RCA CSVs are read-only — nothing is written back to them.** Every tag becomes one
formatted sheet (green = increased, red = decreased, grey = no change; tag-type headers rotated;
`Episode number` = cluster_id frozen as the first column) in a single combined workbook:

`DATA/RCA_cause_behavior_control_actions_ALL_TAGS.xlsx`

## Tags processed (edit the `TAG_CONFIGS` list)
| Tag | Clustered RCA CSV | Control-actions workbook (31-Jul) | Loop PV parquet |
|-----|-------------------|-----------------------------------|-----------------|
| 03LIC_1071 (PVLO) | `RCA_03LIC_1071_clustered_top5.csv` | `…/03LIC_1071_PVLO_episodes_31JUL2026_0846/…` | `03LIC_1071_JAN_2026.parquet` |
| 03LIC_1016 (PVLO) | `RCA_03LIC_1016_clustered_top5.csv` | `…/03LIC_1016_PVLO_episodes_31JUL2026_0850/…` | `03LIC_1016_JAN_2026.parquet` |
| 03LIC_1619 (PVLO) | `RCA_03LIC_1619_clustered_top5.csv` | `…/03LIC_1619_PVLO_episodes_31JUL2026_0852/…` | `03LIC_1619_MAY_2026.parquet` |
| 03PIC_1023 (PVHI) | `RCA_03PIC_1023_clustered_top5.csv` | `…/03PIC_1023_PVHI_episodes_31JUL2026_0837/…` | `03PIC_1023_JULY_2026.parquet` |
| 03TIC_1023 (PVLO) | `RCA_03TIC_1023_clustered_top5.csv` | `…/03TIC_1023_PVLO_episodes_31JUL2026_0843/…` | `03TIC_1023_JAN_2026.parquet` |

## Shared references
| Source | Path | Role |
|--------|------|------|
| KG relevance | `DATA/Full_DB_Merged_Tag_Instrument_Sequences_V17Input 1.xlsx` | `DCS_Path` → per-tag KG tag set |
| PV history (gap-fill) | `DATA/new_rca_pv_op_data/merged_all_historian_tags.parquet` | fills any cause tag missing from a loop export (covers all cause tags) |
| Operating limits | `DATA/operating_limits.csv` | per-tag dead-band for the pattern / direction / ROC call |

**Control-action window** = cluster_start − 30 min → cluster_end + 30 min. Each retained episode's
`Alarm_Timestamp` is the minute-floored `cluster_start_time`, so it maps **exactly** to its cluster.
Every tag is processed with the **identical** engine — only the three inputs per tag differ.


In [1]:
# ═══ C1 · Config: tags to process, shared references, constants ════════════════
import os, shutil
import numpy as np
import pandas as pd
import pyarrow.parquet as _pq
from collections import Counter
from scipy.stats import theilslopes, norm
from IPython.display import display

DATA = "/home/h604827/ControlActions/DATA"
RES  = "/home/h604827/ControlActions/RESULTS"

# ── One entry per alarm tag. 5 tags: clustered RCA CSVs + 31-Jul pipeline workbooks. ─
#   tag          : short id            full         : full PV-less tag (sheet name)
#   input_csv    : clustered RCA file (Alarm_Timestamp + Cause 1-5, one row per RAW cluster)
#                  — READ-ONLY, never modified
#   ca_xlsx      : per-tag clustered-alarms + control_actions workbook (31-Jul run, Stage-3 OFF).
#                  Episodes are subset to control_actions.filtered=True (FI1000 in band + has actions).
#   loop_parquet : per-loop PV/OP export (primary PV source; merged parquet gap-fills)
TAG_CONFIGS = [
    dict(tag="1071", full="03LIC_1071",
         input_csv=f"{DATA}/RCA_03LIC_1071_clustered_top5.csv",
         ca_xlsx=f"{RES}/03LIC_1071_PVLO_episodes_31JUL2026_0846/03LIC_1071_pvlo_alarms_clustered_with_control_actions.xlsx",
         loop_parquet=f"{DATA}/PV-OP_data/03LIC_1071_JAN_2026.parquet"),
    dict(tag="1016", full="03LIC_1016",
         input_csv=f"{DATA}/RCA_03LIC_1016_clustered_top5.csv",
         ca_xlsx=f"{RES}/03LIC_1016_PVLO_episodes_31JUL2026_0850/03LIC_1016_pvlo_alarms_clustered_with_control_actions.xlsx",
         loop_parquet=f"{DATA}/PV-OP_data/03LIC_1016_JAN_2026.parquet"),
    dict(tag="1619", full="03LIC_1619",
         input_csv=f"{DATA}/RCA_03LIC_1619_clustered_top5.csv",
         ca_xlsx=f"{RES}/03LIC_1619_PVLO_episodes_31JUL2026_0852/03LIC_1619_pvlo_alarms_clustered_with_control_actions.xlsx",
         loop_parquet=f"{DATA}/PV-OP_data/03LIC_1619_MAY_2026.parquet"),
    dict(tag="1023P", full="03PIC_1023",
         input_csv=f"{DATA}/RCA_03PIC_1023_clustered_top5.csv",
         ca_xlsx=f"{RES}/03PIC_1023_PVHI_episodes_31JUL2026_0837/03PIC_1023_pvhi_alarms_clustered_with_control_actions.xlsx",
         loop_parquet=f"{DATA}/PV-OP_data/03PIC_1023_JULY_2026.parquet"),
    dict(tag="1023T", full="03TIC_1023",
         input_csv=f"{DATA}/RCA_03TIC_1023_clustered_top5.csv",
         ca_xlsx=f"{RES}/03TIC_1023_PVLO_episodes_31JUL2026_0843/03TIC_1023_pvlo_alarms_clustered_with_control_actions.xlsx",
         loop_parquet=f"{DATA}/PV-OP_data/03TIC_1023_JAN_2026.parquet"),
]

# ── Shared references (identical for every tag) ────────────────────────────────
KG_XLSX        = f"{DATA}/Full_DB_Merged_Tag_Instrument_Sequences_V17Input 1.xlsx"
MERGED_PARQUET = f"{DATA}/new_rca_pv_op_data/merged_all_historian_tags.parquet"
OPLIM_CSV      = f"{DATA}/operating_limits.csv"

# ── Single combined output: one sheet per tag (individual CSVs are NOT edited) ──
OUTPUT_XLSX = f"{DATA}/RCA_cause_behavior_control_actions_ALL_TAGS.xlsx"

# ── Toggles / constants (identical treatment for every tag) ────────────────────
INCLUDE_DIRMAG_MATRIX = True         # per (tag,type) 'dir' + 'avg|step|' columns
WINDOW        = pd.Timedelta(minutes=30)   # control-action window pad each side of the alarm
PRE_MIN       = 180                         # minutes of pre-alarm PV context (SME: look back 3 h)
SMOOTH_MIN    = 5                           # rolling-median smoothing (min)
DEADBAND_FRAC = 0.05                        # dead-band = 5% of operating band ...
NOISE_K       = 2.0                         # ... or k x in-window noise, whichever is larger

# ── Cause-behaviour classifier thresholds (data-calibrated; 3 SME descriptors) ──
#   pattern  : flat | oscillatory | step | sudden_drift | ramp
#   direction: increasing | decreasing | no clear direction
#   roc      : stable | slow | fast   (speed of movement, % of operating band per hour)
# Logic is NET-FIRST: a real net move over 3 h -> DIRECTIONAL (step/ramp/sudden_drift);
# otherwise NON-DIRECTIONAL (oscillatory if it swings with range, else flat).
NET_MULT            = 1.5    # |net move over 3 h| must exceed this x dead-band -> a DIRECTIONAL move
DIR_MIN             = 0.50   # |net| / peak-to-peak range >= this -> it "went somewhere" (ramp/drift);
                             #   below this the excursion cancels out -> oscillatory
FLAT_RANGE_MULT     = 3.0    # non-directional AND peak-to-peak <= this x dead-band -> flat (stable)
OSC_PROM_MULT       = 2.0    # oscillation peak/trough prominence >= this x dead-band ...
OSC_PROM_RANGE_FRAC = 0.15   # ... or this fraction of the window range, whichever is larger
STEP_WIN_MIN        = 10     # a "step" completes within this many minutes
STEP_MIN_JUMP_FRAC  = 0.60   # one <=STEP_WIN_MIN jump holds >= this frac of the net move -> step
DRIFT_LATE_RATIO    = 2.0    # late-half move >= this x early-half move -> sudden_drift (late onset)
ROC_FAST_FRAC_PER_HR = 0.40  # speed >= 40% of operating band / hour -> fast
ROC_SLOW_FRAC_PER_HR = 0.08  # speed >= 8% / hour -> slow (below -> stable)

CAUSE_COLS    = ["Cause 1", "Cause 2", "Cause 3", "Cause 4", "Cause 5"]
NO_ACTIONS_MSG = "No control actions in window"

# ── Shared lookups loaded once ─────────────────────────────────────────────────
_ol = pd.read_csv(OPLIM_CSV).set_index("TAG_NAME")
_kg_paths = pd.read_excel(KG_XLSX, sheet_name=0, usecols=["DCS_Path"])["DCS_Path"].dropna().astype(str)
_kg_token_sets = _kg_paths.apply(lambda p: {t.strip() for t in p.split(",") if t.strip()})

def load_pristine_alarms(input_csv):
    """Raw alarm table (Alarm_Timestamp + Cause 1-5) WITHOUT modifying input_csv.
    If a prior run enriched the CSV in place, read its one-time pristine backup instead."""
    backup = input_csv.replace(".csv", "_ORIGINAL_backup.csv")
    cur = pd.read_csv(input_csv)
    enriched = ("KG_operated_tags" in cur.columns) or ("Episode number" in cur.columns)
    if enriched:
        return pd.read_csv(backup) if os.path.exists(backup) else cur
    if not os.path.exists(backup):
        shutil.copy2(input_csv, backup)          # keep a pristine copy; original left untouched
    return cur

print(f"Tags to process : {[c['full'] for c in TAG_CONFIGS]}")
print(f"Operating limits: {_ol.shape[0]} tags   KG DCS_Path rows: {len(_kg_paths):,}")
print(f"Combined output : {OUTPUT_XLSX}")


Tags to process : ['03LIC_1071', '03LIC_1016', '03LIC_1619', '03PIC_1023', '03TIC_1023']
Operating limits: 40 tags   KG DCS_Path rows: 31,658
Combined output : /home/h604827/ControlActions/DATA/RCA_cause_behavior_control_actions_ALL_TAGS.xlsx


In [2]:
# ═══ C2 · Control-action rendering helpers (identical to the 1071 build) ═══════
_ARROW = {"increase": "↑", "decrease": "↓"}
_UNIT  = {"OP": "%", "SP": ""}
ACT_TYPES = ["OP", "SP"]

def _is_num(x):
    try:
        float(x); return True
    except (ValueError, TypeError):
        return False

def _fmt_val(x):
    return f"{round(float(x), 2):g}"

def _to_float(x):
    try:
        return float(x)
    except (ValueError, TypeError):
        return np.nan

def _action_detail(win):
    """One line per tag (ordered by first action); consecutive same type+direction numeric moves
    collapse to start->end (net, #moves). OP in %, SP in EU; OP/SP/MODE only."""
    df = win[win["Description"].isin(["OP", "SP", "MODE"])].sort_values("VT_Start")
    if df.empty:
        return ""
    first_time = df.groupby("Source")["VT_Start"].min().sort_values()
    lines = []
    for seq, (tag, t0) in enumerate(first_time.items(), 1):
        rows = df[df["Source"] == tag].sort_values("VT_Start").to_dict("records")
        segs = []; i = 0
        while i < len(rows):
            r = rows[i]; desc = r["Description"]; d = r["action_direction"]
            if desc == "MODE":
                segs.append(f"MODE {r['PrevValue']}→{r['Value']}"); i += 1
            elif _is_num(r["PrevValue"]) and _is_num(r["Value"]):
                j = i; last = r
                while (j + 1 < len(rows) and rows[j+1]["Description"] == desc
                       and rows[j+1]["action_direction"] == d
                       and _is_num(rows[j+1]["PrevValue"]) and _is_num(rows[j+1]["Value"])):
                    j += 1; last = rows[j]
                start = float(r["PrevValue"]); end = float(last["Value"]); n = j - i + 1
                net = round(end - start, 2); unit = _UNIT.get(desc, ""); ar = _ARROW.get(d, "")
                mv = f"{n} move" + ("s" if n > 1 else "")
                segs.append(f"{desc}{(' ' + ar) if ar else ''} {_fmt_val(start)}→{_fmt_val(end)} ({net:+g}{unit}, {mv})")
                i = j + 1
            else:
                j = i; last = r
                while (j + 1 < len(rows) and rows[j+1]["Description"] == desc
                       and not (_is_num(rows[j+1]["PrevValue"]) and _is_num(rows[j+1]["Value"]))):
                    j += 1; last = rows[j]
                n = j - i + 1
                segs.append(f"{desc} {r['PrevValue']}→{last['Value']}" + (f" ({n}x)" if n > 1 else ""))
                i = j + 1
        lines.append(f"{seq}. [{t0.strftime('%Y-%m-%d %H:%M')}] {tag} — " + ", then ".join(segs))
    return "\n".join(lines)

def _tagtype_summary(win, kg_set):
    """{(source, type): (direction, avg_abs_step, n_moves)} for KG-kept OP/SP numeric moves.
    Majority direction by #moves; net-change sign breaks ties."""
    df = win[win["Source"].isin(kg_set) & win["Description"].isin(ACT_TYPES)].copy()
    df["pv"] = df["PrevValue"].map(_to_float)
    df["vv"] = df["Value"].map(_to_float)
    df = df[df["pv"].notna() & df["vv"].notna()].sort_values("VT_Start")
    out = {}
    for (src, typ), grp in df.groupby(["Source", "Description"]):
        deltas = (grp["vv"] - grp["pv"]).to_numpy()
        inc, dec = int((deltas > 0).sum()), int((deltas < 0).sum())
        net = float(grp["vv"].iloc[-1] - grp["pv"].iloc[0])
        if inc > dec:
            direction = "increased"
        elif dec > inc:
            direction = "decreased"
        else:
            direction = ("increased" if net > 1e-9 else "decreased" if net < -1e-9 else "no change")
        out[(src, typ)] = (direction, round(float(np.abs(deltas).mean()), 2), len(deltas))
    return out

print("Control-action renderers ready: _action_detail, _tagtype_summary")


Control-action renderers ready: _action_detail, _tagtype_summary


In [3]:
# ═══ C3 · Cause-behaviour engine: denoise -> shape metrics -> pattern/direction/ROC ═
# Three SME-aligned descriptors per cause tag, over the PRE_MIN (=3 h) before the alarm:
#   pattern   : flat | oscillatory | step | sudden_drift | ramp   (SME "Pattern per Cause")
#   direction : increasing | decreasing | no clear direction      (SME "Direction per Cause")
#   roc       : stable | slow | fast                              (SME "ROC Slow/Fast")
# NET-FIRST decision: if the 3 h net move clears NET_MULT x dead-band the tag is DIRECTIONAL
# (step/ramp/sudden_drift by shape); otherwise NON-DIRECTIONAL (oscillatory if it swings across a
# real range, else flat). This keeps noisy *ramps* out of 'oscillatory' while still catching a
# genuine in-band swing such as 03FIC_1085 325<->370.
from scipy.signal import find_peaks

def _theil_slope(sub):
    """Theil-Sen slope (EU per minute) of a time-indexed Series; NaN if too few pts."""
    sub = sub.dropna()
    if len(sub) < 5:
        return np.nan
    x = (sub.index - sub.index[0]).total_seconds().to_numpy() / 60.0
    return theilslopes(sub.to_numpy(), x)[0]

def _mann_kendall(y):
    """Non-parametric monotonic-trend test (kept for QA). Returns (label, p_value)."""
    y = np.asarray(y, dtype=float); y = y[~np.isnan(y)]; n = len(y)
    if n < 6:
        return ('insufficient', np.nan)
    s_stat = 0.0
    for i in range(n - 1):
        s_stat += np.sign(y[i + 1:] - y[i]).sum()
    var = n * (n - 1) * (2 * n + 5) / 18.0
    z = (s_stat - np.sign(s_stat)) / np.sqrt(var)
    p = 2 * (1 - norm.cdf(abs(z)))
    if p < 0.05 and z > 0: return ('increasing', float(p))
    if p < 0.05 and z < 0: return ('decreasing', float(p))
    return ('no-trend', float(p))

def _robust_sigma(resid):
    resid = resid[~np.isnan(resid)]
    if len(resid) < 3:
        return np.nan
    return 1.4826 * np.median(np.abs(resid - np.median(resid)))

def _smooth(s):
    """5-min robust denoise: short-gap interpolate then centred rolling median."""
    return (s.interpolate('linear', limit=SMOOTH_MIN, limit_direction='both')
             .rolling(SMOOTH_MIN, center=True, min_periods=2).median())

def characterize_trend(pv_df, pv_col, cstart):
    """Characterise a cause tag's PV over the PRE_MIN before the alarm (cstart)."""
    anchor = pd.Timestamp(cstart).floor('min')
    grid = pd.date_range(anchor - pd.Timedelta(minutes=PRE_MIN), anchor, freq='1min')
    s = pv_df[pv_col].reindex(grid, method='nearest', tolerance=pd.Timedelta('30s'))
    coverage = float(s.notna().mean())

    out = dict(pv_col=pv_col, coverage=coverage,
               pattern='no_data', direction='no_data', roc='no_data',
               trend_character='no_data', data_quality='insufficient_data', onset_time=pd.NaT)
    if s.notna().sum() < 15:
        return out

    s_s = _smooth(s)
    sigma = _robust_sigma((s - s_s).to_numpy())

    # per-tag operating band + dead-band (5% of band OR k*noise, whichever is larger)
    if pv_col in _ol.index:
        lo, hi = float(_ol.at[pv_col, 'LOWER_LIMIT']), float(_ol.at[pv_col, 'UPPER_LIMIT'])
        op_band = hi - lo if hi > lo else np.nan
    else:
        lo = hi = op_band = np.nan
    # dead-band = 5% of the operating band when limits exist (physical, and immune to the tag's OWN
    # oscillation); only fall back to a noise estimate when limits are unavailable. Using 2*sigma of a
    # fast oscillator inflates the dead-band so its swings look like noise -> mislabelled flat.
    if np.isfinite(op_band):
        deadband = max(DEADBAND_FRAC * op_band, 1e-9)
    elif np.isfinite(sigma):
        deadband = max(NOISE_K * sigma, 1e-9)
    else:
        deadband = max(NOISE_K * float(np.nanstd(s.to_numpy())), 1e-9)

    y = s_s.dropna()
    if len(y) < 15:
        return out
    yv = y.to_numpy(dtype=float)
    dur_min = (y.index[-1] - y.index[0]).total_seconds() / 60.0

    # ── shape metrics ─────────────────────────────────────────────────────────
    slope_full = _theil_slope(y)
    net_full   = slope_full * dur_min if np.isfinite(slope_full) else np.nan
    v_start, v_alarm = float(yv[0]), float(yv[-1])
    rng = float(np.nanmax(yv) - np.nanmin(yv))                 # peak-to-peak of smoothed
    tv  = float(np.abs(np.diff(yv)).sum())                     # total variation (path length)
    directionality = (abs(net_full) / tv) if (tv > 1e-9 and np.isfinite(net_full)) else 0.0
    span = op_band if np.isfinite(op_band) else max(rng, 1e-9)

    # prominent turning points of the smoothed series (QA / oscillation evidence)
    prom = max(OSC_PROM_MULT * deadband, OSC_PROM_RANGE_FRAC * rng)
    n_peaks   = len(find_peaks(yv,  prominence=prom)[0])
    n_troughs = len(find_peaks(-yv, prominence=prom)[0])
    n_turns   = n_peaks + n_troughs

    # early vs late halves -> ramp (spread) vs sudden_drift (late onset)
    mid = anchor - pd.Timedelta(minutes=PRE_MIN // 2)
    slope_early = _theil_slope(y.loc[:mid]); net_early = slope_early * (PRE_MIN/2) if np.isfinite(slope_early) else np.nan
    slope_late  = _theil_slope(y.loc[mid:]); net_late  = slope_late  * (PRE_MIN/2) if np.isfinite(slope_late)  else np.nan

    # largest change over any <= STEP_WIN_MIN interval -> step (level shift) evidence
    step_win = int(min(STEP_WIN_MIN, len(yv) - 1))
    diffs_win = yv[step_win:] - yv[:-step_win] if step_win >= 1 else np.array([0.0])
    jump = float(diffs_win[int(np.argmax(np.abs(diffs_win)))]) if len(diffs_win) else 0.0
    step_share = abs(jump) / max(tv, 1e-9)        # share of the whole 3 h path taken by one short jump

    # 30-min segment slopes -> oscillation speed
    seg_slopes = []
    for k in range(max(1, PRE_MIN // 30)):
        seg = y.loc[anchor - pd.Timedelta(minutes=30*(k+1)): anchor - pd.Timedelta(minutes=30*k)]
        ms = _theil_slope(seg)
        if np.isfinite(ms):
            seg_slopes.append(ms)

    mk_label, mk_p = _mann_kendall(yv)

    # ── NET-FIRST, MULTI-WINDOW classification ────────────────────────────────
    # Judge the move INTO the alarm on the full 3 h AND on shorter recent windows, so a tag that
    # wanders for 2 h then drives into the alarm in the last hour reads as 'sudden_drift' (late onset)
    # rather than being diluted to 'oscillatory'. A tag that wanders throughout stays 'oscillatory'.
    net_significant   = np.isfinite(net_full) and abs(net_full) > NET_MULT * deadband
    range_significant = rng > FLAT_RANGE_MULT * deadband
    net_over_range    = (abs(net_full) / rng) if (np.isfinite(net_full) and rng > 1e-9) else 0.0

    def _win_move(w):
        seg = y.loc[anchor - pd.Timedelta(minutes=w): anchor].dropna()
        sl = _theil_slope(seg)
        if not np.isfinite(sl) or len(seg) < 5:
            return (np.nan, 0.0, 0.0)
        nseg = sl * w
        rseg = float(seg.to_numpy().max() - seg.to_numpy().min())
        nor = (abs(nseg) / rseg) if rseg > 1e-9 else 0.0
        return (nseg, rseg, nor)

    DIR_WINDOWS = [PRE_MIN, PRE_MIN // 2, 60]                 # 180, 90, 60 min (context -> recent)
    moves = {w: _win_move(w) for w in DIR_WINDOWS}
    def _dir_ok(w):
        nseg, rseg, nor = moves[w]
        return (np.isfinite(nseg) and abs(nseg) > NET_MULT * deadband
                and rseg > deadband and nor >= DIR_MIN)
    full_dir    = _dir_ok(PRE_MIN)
    recent_dirs = [w for w in DIR_WINDOWS if w < PRE_MIN and _dir_ok(w)]

    is_step = abs(jump) > NET_MULT * deadband and step_share >= STEP_MIN_JUMP_FRAC
    if is_step:
        pattern, decide_slope = 'step', jump / max(step_win, 1)
        direction = 'increasing' if jump > 0 else 'decreasing'
    elif full_dir:
        pattern, net_d = 'ramp', moves[PRE_MIN][0]            # steady directional move across 3 h
        decide_slope = net_d / PRE_MIN
        direction = 'increasing' if net_d > 0 else 'decreasing'
    elif recent_dirs:
        w0 = min(recent_dirs)                                 # shortest = most recent onset
        pattern, net_d = 'sudden_drift', moves[w0][0]         # flat/wander early, drives in late
        decide_slope = net_d / w0
        direction = 'increasing' if net_d > 0 else 'decreasing'
    elif range_significant or net_significant:
        pattern = 'oscillatory'                              # moves but the excursion cancels out
        decide_slope = float(np.median(np.abs(seg_slopes))) if seg_slopes else (slope_full or 0.0)
        direction = 'no clear direction'
    else:
        pattern, decide_slope, direction = 'flat', 0.0, 'no clear direction'

    # ── ROC (movement speed, as % of operating band per hour) ─────────────────
    if pattern == 'flat':
        roc, rate_frac_per_hr = 'stable', 0.0
    else:
        speed = abs(decide_slope) if np.isfinite(decide_slope) else 0.0
        rate_frac_per_hr = speed * 60.0 / span
        roc = ('fast' if rate_frac_per_hr >= ROC_FAST_FRAC_PER_HR
               else 'slow' if rate_frac_per_hr >= ROC_SLOW_FRAC_PER_HR else 'stable')

    # ── onset (trough before a rise / peak before a fall) ─────────────────────
    onset_time, from_onset_min, from_onset_eu = pd.NaT, np.nan, np.nan
    if direction in ('increasing', 'decreasing'):
        onset_time = y.idxmin() if direction == 'increasing' else y.idxmax()
        from_onset_min = (anchor - onset_time).total_seconds() / 60.0
        from_onset_eu  = v_alarm - float(y.loc[onset_time])

    pos = ('below_low' if (np.isfinite(lo) and v_alarm < lo)
           else 'above_high' if (np.isfinite(hi) and v_alarm > hi)
           else 'within' if np.isfinite(lo) else 'unknown')

    out.update(
        data_quality=('ok' if coverage >= 0.5 else 'sparse'),
        pattern=pattern, direction=direction, roc=roc, trend_character=pattern,
        value_at_alarm=v_alarm, value_start=v_start,
        net_full_eu=net_full, rng_eu=rng, total_variation=tv, directionality=directionality,
        slope_full=slope_full, slope_early=slope_early, slope_late=slope_late,
        net_early=net_early, net_late=net_late, jump_eu=jump, step_share=step_share,
        n_turns=n_turns, rate_frac_per_hr=rate_frac_per_hr,
        net_x_deadband=(abs(net_full)/deadband if (np.isfinite(net_full) and deadband > 0) else np.nan),
        range_x_deadband=(rng/deadband if deadband > 0 else np.nan),
        net_over_range=net_over_range,
        mk_trend=mk_label, mk_p=mk_p,
        onset_time=onset_time, from_onset_min=from_onset_min, from_onset_eu=from_onset_eu,
        sigma_noise=sigma, op_band=op_band, deadband_eu=deadband, at_alarm_vs_limits=pos,
    )
    return out

def _behavior_word(r):
    """Compact 3-part SME label written next to the tag: 'pattern | direction | ROC'."""
    p = r["pattern"]
    if p == "no_data":
        return "no data"
    if p == "flat":
        return "flat | no clear direction | stable"
    return f'{p} | {r["direction"]} | {r["roc"]}'

print("Cause-behaviour engine ready: characterize_trend(pv_df, pv_col, cstart), _behavior_word(r)")
print("  NET-FIRST: pattern in {flat, oscillatory, step, sudden_drift, ramp} | direction | roc in {stable, slow, fast}")


Cause-behaviour engine ready: characterize_trend(pv_df, pv_col, cstart), _behavior_word(r)
  NET-FIRST: pattern in {flat, oscillatory, step, sudden_drift, ramp} | direction | roc in {stable, slow, fast}


In [4]:
# ═══ C4 · Per-tag pipeline: KG set -> clusters -> control actions -> PV -> trends ═
def kg_tags_for(full):
    """Union of all tags on every DCS_Path that contains the target tag."""
    tags = set()
    for ts in _kg_token_sets[_kg_token_sets.apply(lambda x: full in x)]:
        tags |= ts
    return tags

def _clusters_and_actions(ca_xlsx):
    """cluster bounds (CB) + de-duplicated control actions (CA_DEDUP) + retained-cluster set
    (RETAINED = control_actions.filtered==True) from one workbook."""
    ac = pd.read_excel(ca_xlsx, sheet_name="alarm_clusters")
    ac["cluster_start_time"] = pd.to_datetime(ac["cluster_start_time"])
    ac["cluster_end_time"]   = pd.to_datetime(ac["cluster_end_time"])
    CB = (ac.groupby("cluster_id")
            .agg(cstart=("cluster_start_time", "min"), cend=("cluster_end_time", "max"))
            .sort_values("cstart"))
    CA = pd.read_excel(ca_xlsx, sheet_name="control_actions",
                       usecols=["cluster_id", "Source", "Description", "action_direction",
                                "PrevValue", "Value", "VT_Start", "filtered"])
    CA["VT_Start"] = pd.to_datetime(CA["VT_Start"])
    CA = CA[CA["Source"].notna()].copy()
    # filtered==True marks clusters the pipeline kept (Stage-1 duration + Stage-2 FI1000 band +
    # Stage-4 has OP/SP/MODE actions; Stage-3 quiet-window was OFF in the 31-Jul runs).
    retained_cids = set(CA.loc[CA["filtered"] == True, "cluster_id"].astype(int).unique())
    CA_DEDUP = CA.drop_duplicates(subset=["Source", "Description", "VT_Start", "PrevValue", "Value"])
    return CB, CA_DEDUP, retained_cids

def _load_pv(needed_pv, loop_parquet):
    """PV history for needed cause tags: loop export primary, merged historian gap-fill."""
    def _read_avail(path, cols):
        names = set(_pq.read_schema(path).names)
        have = [c for c in cols if c in names]
        if not have:
            return None
        df = pd.read_parquet(path, columns=["TimeStamp", *have])
        df = df.dropna(subset=["TimeStamp"]).set_index("TimeStamp").sort_index()
        return df[~df.index.duplicated(keep="last")]
    pv = _read_avail(loop_parquet, needed_pv) if (loop_parquet and os.path.exists(loop_parquet)) else None
    if pv is None:
        pv = pd.DataFrame(index=pd.DatetimeIndex([], name="TimeStamp"))
    missing = [c for c in needed_pv if c not in pv.columns]
    if missing:
        m = _read_avail(MERGED_PARQUET, missing)
        if m is not None:
            pv = pv.join(m, how="outer")
    for c in needed_pv:
        if c not in pv.columns:
            pv[c] = np.nan
    return pv.astype("float64").sort_index(), missing

def process_tag(cfg):
    """Run the full 1071-style enrichment for one tag; return the assembled sheet + diagnostics."""
    full = cfg["full"]
    alarms = load_pristine_alarms(cfg["input_csv"])
    alarms["Alarm_Timestamp"] = pd.to_datetime(alarms["Alarm_Timestamp"])
    alarms = alarms.sort_values("Alarm_Timestamp").reset_index(drop=True)
    KG_TAGS = kg_tags_for(full)

    # clusters + EXACT minute alarm->cluster mapping (nearest kept only as a safety net)
    CB, CA_DEDUP, retained_cids = _clusters_and_actions(cfg["ca_xlsx"])
    CB["cstart_min"] = CB["cstart"].dt.floor("min")
    cstart_to_cid = {t: cid for cid, t in CB["cstart_min"].items()}
    cs = CB["cstart"].to_numpy().astype("datetime64[ns]")
    cid_arr = CB.index.to_numpy()
    def _win(alarm_ts):
        tmin = pd.Timestamp(alarm_ts).floor("min")
        cid = cstart_to_cid.get(tmin)
        if cid is not None:
            return (CB.at[cid, "cstart"] - WINDOW, CB.at[cid, "cend"] + WINDOW, int(cid), "cluster")
        i = int(np.abs(cs - np.datetime64(tmin, "ns")).argmin())
        return (CB.iloc[i]["cstart"] - WINDOW, CB.iloc[i]["cend"] + WINDOW, int(cid_arr[i]), "nearest")
    _w = alarms["Alarm_Timestamp"].apply(_win)
    alarms["win_start"]  = _w.apply(lambda x: x[0])
    alarms["win_end"]    = _w.apply(lambda x: x[1])
    alarms["cluster_id"] = _w.apply(lambda x: x[2])
    alarms["win_method"] = _w.apply(lambda x: x[3])

    # Subset the clustered RCA CSV (one row per RAW cluster) to only the use-case episodes:
    # clusters flagged filtered=True in the workbook (FI1000 in band + carry OP/SP/MODE actions).
    n_input = len(alarms)
    alarms = alarms[alarms["cluster_id"].isin(retained_cids)].reset_index(drop=True)
    n_retained = len(alarms)

    # per-alarm control-action fields (absolute-time window on de-dup actions, KG-filtered)
    kept_list, elim_list, detail_list, ttsummary_list = [], [], [], []
    tagtype_freq = Counter()
    for _, row in alarms.iterrows():
        win = CA_DEDUP[(CA_DEDUP["VT_Start"] >= row["win_start"]) &
                       (CA_DEDUP["VT_Start"] <= row["win_end"])]
        operated = sorted(win["Source"].unique())
        if not operated:
            kept_list.append(NO_ACTIONS_MSG); elim_list.append(""); detail_list.append("")
            ttsummary_list.append({}); continue
        kept = [t for t in operated if t in KG_TAGS]
        elim = [t for t in operated if t not in KG_TAGS]
        kept_list.append(", ".join(kept)); elim_list.append(", ".join(elim))
        detail_list.append(_action_detail(win[win["Source"].isin(KG_TAGS)]))
        summ = _tagtype_summary(win, KG_TAGS); ttsummary_list.append(summ); tagtype_freq.update(summ.keys())
    alarms["KG_operated_tags"] = kept_list
    alarms["eliminated_tags"]  = elim_list
    alarms["action_details"]   = detail_list

    # PV history for the cause tags, then characterise each cause behaviour (per-tag cache)
    needed_pv = sorted({v.strip() for c in CAUSE_COLS for v in alarms[c].dropna().astype(str)
                        if v.strip().endswith(".PV")})
    pv_df, gapfilled = _load_pv(needed_pv, cfg["loop_parquet"])
    cache = {}
    def _char(pv_col, anchor):
        k = (pv_col, pd.Timestamp(anchor).floor("min"))
        if k not in cache:
            cache[k] = characterize_trend(pv_df, pv_col, anchor)
        return cache[k]

    enriched = alarms.copy()
    metric_rows = []
    for i, row in alarms.iterrows():
        anchor = row["Alarm_Timestamp"]
        for slot, col in enumerate(CAUSE_COLS, start=1):
            tag = row[col]
            if not isinstance(tag, str) or not tag.strip().endswith(".PV"):
                continue
            tag = tag.strip()
            res = _char(tag, anchor)
            metric_rows.append(dict(alarm_row=i, alarm_timestamp=anchor, cause_slot=slot,
                                    cause_col=col, **res))
            enriched.at[i, col] = f"{tag} [{_behavior_word(res)}]"
    metrics = pd.DataFrame(metric_rows)
    if len(metrics):
        metrics["behavior"] = metrics.apply(_behavior_word, axis=1)

    # per (tag,type) direction + avg|step| columns (most-operated first)
    dirmag_cols = []
    if INCLUDE_DIRMAG_MATRIX:
        order = [k for k, _ in sorted(tagtype_freq.items(), key=lambda kv: (-kv[1], kv[0]))]
        for (src, typ) in order:
            dcol, mcol = f"{src}.{typ} dir", f"{src}.{typ} avg|step|"
            enriched[dcol] = [s.get((src, typ), ("", np.nan, 0))[0] for s in ttsummary_list]
            enriched[mcol] = [s.get((src, typ), ("", np.nan, 0))[1] for s in ttsummary_list]
            dirmag_cols += [dcol, mcol]

    # assemble the sheet: Episode number (=cluster_id) first, then causes, meta, dir/mag matrix
    meta_cols = ["KG_operated_tags", "eliminated_tags", "action_details"]
    out = enriched[["Alarm_Timestamp"] + CAUSE_COLS + meta_cols + dirmag_cols].copy()
    out.insert(0, "Episode number", enriched["cluster_id"].astype("Int64").values)
    out["Alarm_Timestamp"] = pd.to_datetime(out["Alarm_Timestamp"]).dt.strftime("%Y-%m-%d %H:%M:%S")

    return dict(full=full, out=out, metrics=metrics, pv_df=pv_df,
                n_alarms=len(alarms), n_cluster=int((alarms["win_method"] == "cluster").sum()),
                n_input=n_input, n_retained=n_retained,
                n_with_actions=sum(1 for k in kept_list if k != NO_ACTIONS_MSG),
                n_dirmag=len(dirmag_cols) // 2, needed_pv=needed_pv, gapfilled=gapfilled)

print("process_tag(cfg) ready.")


process_tag(cfg) ready.


In [5]:
# ═══ C5 · Build the combined workbook — one coloured sheet per tag ═════════════
# Same formatting as the earlier single-tag deliverable, applied to every sheet:
#   direction cells green=increased / red=decreased / grey=no change; tag-type headers rotated
#   vertical; action_details wrapped; Episode number (=cluster_id) + top row frozen.
import openpyxl
from openpyxl.styles import PatternFill, Alignment, Font

DIR_FILL = {
    "increased": PatternFill("solid", fgColor="C6EFCE"),   # green
    "decreased": PatternFill("solid", fgColor="FFC7CE"),   # red
    "no change": PatternFill("solid", fgColor="D9D9D9"),   # grey
}
VERT = Alignment(text_rotation=90, vertical="bottom", horizontal="center")

def format_sheet(ws):
    """Colour/rotate/freeze one worksheet in place; returns (tag-type pairs, coloured cells)."""
    hdr = {ws.cell(row=1, column=c).value: c for c in range(1, ws.max_column + 1)}
    dirmag_pairs = []
    for h in hdr:
        if isinstance(h, str) and h.endswith(" dir"):
            mh = f"{h[:-4]} avg|step|"
            if mh in hdr:
                dirmag_pairs.append((h, mh))
    for dcol, mcol in dirmag_pairs:                       # rotate + bold tag-type headers
        for name in (dcol, mcol):
            cell = ws.cell(row=1, column=hdr[name]); cell.alignment = VERT; cell.font = Font(bold=True)
        ws.column_dimensions[ws.cell(row=1, column=hdr[dcol]).column_letter].width = 11
        ws.column_dimensions[ws.cell(row=1, column=hdr[mcol]).column_letter].width = 9
    for name, width in [("KG_operated_tags", 26), ("eliminated_tags", 22), ("action_details", 72)]:
        if name in hdr:
            ws.column_dimensions[ws.cell(row=1, column=hdr[name]).column_letter].width = width
    if "action_details" in hdr:
        for r in range(2, ws.max_row + 1):
            ws.cell(row=r, column=hdr["action_details"]).alignment = Alignment(wrap_text=True, vertical="top")
    n_filled = 0                                          # colour every operated tag-type cell
    for dcol, mcol in dirmag_pairs:
        dci, mci = hdr[dcol], hdr[mcol]
        for r in range(2, ws.max_row + 1):
            fill = DIR_FILL.get(ws.cell(row=r, column=dci).value)
            if fill:
                ws.cell(row=r, column=dci).fill = fill
                ws.cell(row=r, column=mci).fill = fill
                n_filled += 1
    ws.row_dimensions[1].height = 120
    if "Episode number" in hdr:
        ws.column_dimensions[ws.cell(row=1, column=hdr["Episode number"]).column_letter].width = 11
    ws.freeze_panes = "B2"
    return len(dirmag_pairs), n_filled

# process every configured tag, then write + format one sheet each
RESULTS_BY_TAG = {}
for cfg in TAG_CONFIGS:
    res = process_tag(cfg)
    RESULTS_BY_TAG[cfg["full"]] = res
    warn = "" if res["n_cluster"] == res["n_alarms"] else f"  ⚠ {res['n_alarms'] - res['n_cluster']} not exact-matched"
    print(f"{res['full']:12s}: {res['n_retained']:>4}/{res['n_input']:>4} episodes kept (filtered=True) | "
          f"{res['n_with_actions']:>4} with actions | "
          f"{res['n_dirmag']:>2} tag-type pairs | {res['out'].shape[1]} cols{warn}")

with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    for full, res in RESULTS_BY_TAG.items():
        res["out"].to_excel(writer, sheet_name=full[:31], index=False)

wb = openpyxl.load_workbook(OUTPUT_XLSX)
for full, res in RESULTS_BY_TAG.items():
    pairs, filled = format_sheet(wb[full[:31]])
    print(f"  sheet {full[:31]:12s}: {res['out'].shape[0]:>4} rows x {res['out'].shape[1]:>3} cols  "
          f"{pairs} pairs, {filled} coloured cells")
wb.save(OUTPUT_XLSX)
print(f"\nWrote combined workbook -> {OUTPUT_XLSX}   ({len(RESULTS_BY_TAG)} sheets)")

# QA: cause-behaviour pattern distribution across all tags (should be spread, not one class)
_allm = pd.concat([r["metrics"] for r in RESULTS_BY_TAG.values() if len(r["metrics"])],
                  ignore_index=True)
_allm = _allm[_allm["pattern"] != "no_data"]
print("\nCause-behaviour pattern distribution (all tags, all cause slots):")
print((_allm["pattern"].value_counts(normalize=False).to_frame("count")
       .assign(pct=lambda d: (100 * d["count"] / d["count"].sum()).round(1))).to_string())


03LIC_1071  :  272/ 539 episodes kept (filtered=True) |  272 with actions | 40 tag-type pairs | 90 cols
03LIC_1016  :  115/ 594 episodes kept (filtered=True) |  115 with actions | 21 tag-type pairs | 52 cols
03LIC_1619  :  116/ 276 episodes kept (filtered=True) |  116 with actions |  8 tag-type pairs | 26 cols
03PIC_1023  :   85/ 113 episodes kept (filtered=True) |   85 with actions |  6 tag-type pairs | 22 cols
03TIC_1023  :  383/ 765 episodes kept (filtered=True) |  383 with actions | 16 tag-type pairs | 42 cols
  sheet 03LIC_1071  :  272 rows x  90 cols  40 pairs, 647 coloured cells
  sheet 03LIC_1016  :  115 rows x  52 cols  21 pairs, 219 coloured cells
  sheet 03LIC_1619  :  116 rows x  26 cols  8 pairs, 213 coloured cells
  sheet 03PIC_1023  :   85 rows x  22 cols  6 pairs, 112 coloured cells
  sheet 03TIC_1023  :  383 rows x  42 cols  16 pairs, 173 coloured cells

Wrote combined workbook -> /home/h604827/ControlActions/DATA/RCA_cause_behavior_control_actions_ALL_TAGS.xlsx   (5 s

In [6]:
# ═══ C6 · Visual validation — plot the 3-h pre-alarm window + a gallery per pattern ═
import plotly.graph_objects as go

def _vmark(fig, xts, color, text, dash="dot"):
    fig.add_shape(type="line", x0=xts, x1=xts, y0=0, y1=1, yref="paper",
                  line=dict(color=color, dash=dash, width=1.5))
    fig.add_annotation(x=xts, y=1.02, yref="paper", yanchor="bottom", text=text,
                       showarrow=False, font=dict(color=color, size=11))

def plot_cause_trend(full, i):
    """Raw + smoothed PV for row i of tag `full`'s metrics, with Theil-Sen line, dead-band,
    onset and operating limits. Title carries the pattern | direction | ROC verdict."""
    res = RESULTS_BY_TAG[full]; metrics = res["metrics"]; pv_df = res["pv_df"]
    r = metrics.iloc[i]
    anchor = pd.Timestamp(r["alarm_timestamp"]).floor("min")
    grid = pd.date_range(anchor - pd.Timedelta(minutes=PRE_MIN), anchor, freq="1min")
    s = pv_df[r["pv_col"]].reindex(grid, method="nearest", tolerance=pd.Timedelta("30s"))
    s_s = _smooth(s)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=grid, y=s, mode="lines", name="raw PV",
                             line=dict(color="lightgray", width=1)))
    fig.add_trace(go.Scatter(x=grid, y=s_s, mode="lines", name="smoothed (5-min median)",
                             line=dict(color="royalblue", width=2)))
    base = s_s.dropna()
    if len(base) and np.isfinite(r.get("slope_full", np.nan)):
        b0 = base.iloc[0]; xm = (grid - grid[0]).total_seconds() / 60.0
        fig.add_trace(go.Scatter(x=grid, y=b0 + r["slope_full"] * xm, mode="lines",
                                 name="Theil-Sen (3 h)", line=dict(color="firebrick", dash="dash")))
        if np.isfinite(r.get("deadband_eu", np.nan)):
            for sgn in (+1, -1):
                fig.add_trace(go.Scatter(x=grid, y=[b0 + sgn * r["deadband_eu"]] * len(grid),
                                         mode="lines", line=dict(color="gray", dash="dot", width=1),
                                         name="dead-band", showlegend=(sgn == 1)))
    if r["pv_col"] in _ol.index:
        for lim, nm in [(_ol.at[r["pv_col"], "LOWER_LIMIT"], "low lim"),
                        (_ol.at[r["pv_col"], "UPPER_LIMIT"], "high lim")]:
            fig.add_hline(y=float(lim), line=dict(color="orange", dash="dot", width=1),
                          annotation_text=nm)
    if not pd.isna(r.get("onset_time", pd.NaT)):
        _vmark(fig, pd.Timestamp(r["onset_time"]), "green", "onset")
    _vmark(fig, anchor, "black", "alarm", dash="solid")
    fig.update_layout(
        title=(f"{full} · row {r['alarm_row']} · {r['cause_col']} · {r['pv_col']} · "
               f"alarm {anchor:%Y-%m-%d %H:%M}<br><sub><b>{_behavior_word(r)}</b> · "
               f"Δ3h={r.get('net_full_eu', float('nan')):+.2f} · turns={int(r.get('n_turns', 0))} · "
               f"dirn-idx={r.get('directionality', float('nan')):.2f} · cov {r['coverage']:.0%}</sub>"),
        height=460, template="plotly_white", xaxis_title="time",
        yaxis_title=r["pv_col"], legend=dict(orientation="h", y=-0.2))
    return fig

def show_examples_per_pattern(n_per=3, tags=None, min_cov=0.5):
    """Plot up to n_per DIVERSE examples of every pattern (across all processed tags), so the
    within-pattern variations (direction, speed) are visible side by side."""
    tags = tags or list(RESULTS_BY_TAG)
    buckets = {}
    for full in tags:
        m = RESULTS_BY_TAG[full].get("metrics")
        if m is None or not len(m):
            continue
        for i in range(len(m)):
            r = m.iloc[i]
            if r["pattern"] == "no_data":
                continue
            buckets.setdefault(r["pattern"], []).append(
                (full, i, float(r.get("coverage", 0.0)), r["direction"], r["roc"]))
    order = ["ramp", "sudden_drift", "step", "oscillatory", "flat"]
    for pat in order + [p for p in buckets if p not in order]:
        items = buckets.get(pat, [])
        if not items:
            print(f"— {pat}: (none)"); continue
        items = sorted(items, key=lambda t: (-(t[2] >= min_cov), -t[2]))   # good coverage first
        picked, seen = [], set()
        for it in items:                                                    # diverse (dir, roc) combos
            k = (it[3], it[4])
            if k not in seen:
                seen.add(k); picked.append(it)
            if len(picked) >= n_per:
                break
        for it in items:                                                    # top up if still short
            if len(picked) >= n_per: break
            if it not in picked: picked.append(it)
        combos = ", ".join(sorted({f"{d}/{r}" for _, _, _, d, r in picked}))
        print(f"— {pat}: {len(items)} total; showing {len(picked)} ({combos})")
        for full, i, *_ in picked:
            plot_cause_trend(full, i).show()

show_examples_per_pattern(n_per=3)
print("\nUse plot_cause_trend(full, i) for any tag/row, or show_examples_per_pattern(n_per=N).")


— ramp: 1592 total; showing 3 (decreasing/slow, increasing/fast, increasing/slow)


— sudden_drift: 1171 total; showing 3 (decreasing/fast, increasing/fast, increasing/slow)


— step: 19 total; showing 3 (decreasing/fast, increasing/fast)


— oscillatory: 1826 total; showing 3 (no clear direction/fast, no clear direction/slow, no clear direction/stable)


— flat: 116 total; showing 3 (no clear direction/stable)



Use plot_cause_trend(full, i) for any tag/row, or show_examples_per_pattern(n_per=N).


In [ ]:
# ═══ C7 · Episode-64 spot check (03LIC_1071): confirm 03FIC_1085 reads as oscillatory ═
_full = "03LIC_1071"
_res  = RESULTS_BY_TAG[_full]
_out, _m = _res["out"], _res["metrics"]
_rows = _out.index[_out["Episode number"] == 64].tolist()
print(f"Episode 64 -> alarm row(s) {_rows}")
if not _rows:
    print("  Episode 64 is not in the retained (filtered=True) set for 03LIC_1071 — skipping spot check.")
else:
    for _c in CAUSE_COLS:
        print(f"  {_c}: {_out.loc[_rows[0], _c]}")

    _sub = _m[_m["alarm_row"].isin(_rows)][
        ["cause_col", "pv_col", "pattern", "direction", "roc",
         "net_over_range", "range_x_deadband", "rng_eu", "deadband_eu", "net_full_eu", "coverage"]]
    display(_sub.round(3))
    for _i in _sub.index:
        plot_cause_trend(_full, int(_i)).show()


Episode 64 -> alarm row(s) [2]
  Cause 1: 03FIC_1085.PV [oscillatory | no clear direction | slow]
  Cause 2: 03FI_3418.PV [ramp | decreasing | slow]
  Cause 3: 03TI_1015.PV [sudden_drift | decreasing | slow]
  Cause 4: 02FI_1000.PV [oscillatory | no clear direction | fast]
  Cause 5: 03TIC_1092.PV [oscillatory | no clear direction | slow]


,cause_col,pv_col,pattern,direction,roc,net_over_range,range_x_deadband,rng_eu,deadband_eu,net_full_eu,coverage
10,Cause 1,03FIC_1085.PV,oscillatory,no clear direction,slow,0.110,23.499,39.674,1.688,-4.364,1.0
11,Cause 2,03FI_3418.PV,ramp,decreasing,slow,0.557,3.882,2.488,0.641,-1.385,1.0
12,Cause 3,03TI_1015.PV,sudden_drift,decreasing,slow,0.363,24.252,1.905,0.079,-0.691,1.0
13,Cause 4,02FI_1000.PV,oscillatory,no clear direction,fast,0.365,27.427,0.334,0.012,-0.122,1.0
14,Cause 5,03TIC_1092.PV,oscillatory,no clear direction,slow,0.000,25.314,2.289,0.090,0.000,1.0


   cause_col         pv_col       pattern           direction   roc  net_over_range  range_x_deadband  rng_eu  deadband_eu  net_full_eu  coverage
10   Cause 1  03FIC_1085.PV   oscillatory  no clear direction  slow           0.110            23.499  39.674        1.688       -4.364       1.0
11   Cause 2   03FI_3418.PV          ramp          decreasing  slow           0.557             3.882   2.488        0.641       -1.385       1.0
12   Cause 3   03TI_1015.PV  sudden_drift          decreasing  slow           0.363            24.252   1.905        0.079       -0.691       1.0
13   Cause 4   02FI_1000.PV   oscillatory  no clear direction  fast           0.365            27.427   0.334        0.012       -0.122       1.0
14   Cause 5  03TIC_1092.PV   oscillatory  no clear direction  slow           0.000            25.314   2.289        0.090        0.000       1.0
